# Atividade Feature Selection Algoritmo Genético e PSO para Classificação diferentes variedades de trigo utilizando modelos de aprendizagem de maquina

---

**Curso** : Pos graduação em Ciencia de Dados - Otimizacao e computacao evolucionaria

**Alunos** : Marcelo Manoel dos Santos Miranda, Flavius Raymundo Arruda Sodré, Robson Florencio Correia

**Orientador**: Prof. Adiel Filho


---

**O banco de dados:** Medições das propriedades geométricas de grãos pertencentes a três variedades diferentes de trigo. Uma técnica de raios X suaves e o pacote GRAINS foram usados ​​para construir todos os sete atributos de valor real.

**Informações adicionais:** O grupo analisado era composto por grãos pertencentes a três variedades diferentes de trigo: Kama, Rosa e Canadian, 70 elementos de cada variedade, selecionados aleatoriamente para o experimento. A visualização de alta qualidade da estrutura interna do grão foi obtida utilizando uma técnica de raios X suaves. Essa técnica é não destrutiva e consideravelmente mais barata do que outras técnicas de imagem mais sofisticadas, como microscopia eletrônica de varredura ou tecnologia a laser. As imagens foram registradas em placas de raios X KODAK de 13x18 cm. Os estudos foram conduzidos utilizando grãos de trigo colhidos por colheitadeira em campos experimentais explorados no Instituto de Agrofísica da Academia Polonesa de Ciências em Lublin. O conjunto de dados pode ser utilizado para tarefas de classificação e análise de agrupamentos.


# Imports e Setup

Vamos adicionar os imports necessários para classificação e as bibliotecas de otimização.

In [ ]:
!pip install deap
# !pip install pyswarms # Manteremos a implementação nativa para simplificar a integração com a estrutura do K-Fold
!pip install scikit-learn

import numpy as np
import pandas as pd
import time
from scipy import stats # Necessário para o Teste t
from deap import base, creator, tools, algorithms
from sklearn.model_selection import train_test_split, KFold # KFold para avaliação robusta
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression # Classificador 1
from sklearn.tree import DecisionTreeClassifier # Classificador 2
from sklearn.ensemble import RandomForestClassifier # Classificador 3
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# --- Configurações Gerais e do Problema ---
N_FEATURES = 7 # Número de atributos
N_CLASSES = 3 # Número de classes (variedades de sementes: Kama, Rosa, Canadian)
RANDOM_SEED = 42

# --- Configurações da Otimização (GA e PSO) ---
# O número de gerações do GA deve ser igual ao número de iterações do PSO
N_GENERATIONS = 50
SWARM_SIZE = 50 # Tamanho da População / Enxame
CX_RATE = 0.6
MUT_RATE = 0.15
W_PSO = 0.5
C1_PSO = 1
C2_PSO = 2
N_RUNS = 30 # Número de repetições para a análise estatística
#Ajustamos os valores das configuracoes buscando obter os melhores resultados

# --- Classificador Base para a Função de Aptidão (Usaremos Random Forest) ---
# Foi escolhido o Random Forest por ser geralmente mais robusto.
CLASSIFIER_FITNESS = RandomForestClassifier(random_state=RANDOM_SEED)
# Adicionalmente, se o arquivo seeds.txt não funcionar, você pode usar a versão
# do UCI salva como CSV na pasta 'datasets' do GitHub
# !wget https://archive.ics.uci.edu/ml/machine-learning-databases/00236/seeds_dataset.txt

#1. Setup e Pré-processamento de Dados:

Esta seção cobre o carregamento dos dados "Seeds" do UCI e a preparação para o K-means.

In [ ]:
# Montar o Drive e carregar os dados (Ajustar o caminho conforme necessário)
from google.colab import drive
try:
    drive.mount('/content/drive')
except:
    print("A montagem do Google Drive falhou. Garanta que você está no ambiente Google Colab.")

def load_and_preprocess_data():
    # Caminho do arquivo no Drive (MANTIDO o caminho original)
    data_path = "/content/drive/MyDrive/Otimização/seeds_dataset.txt"
    try:
        # Lendo com separador de um ou mais espaços (\s+) e sem cabeçalho
        data = pd.read_csv(data_path, sep=r'\s+', header=None, engine='python')
        # As primeiras 7 colunas são as features (X)
        X = data.iloc[:, :N_FEATURES].values
        # A última coluna é o rótulo de classe (Y), subtraindo 1 para ter rótulos de 0, 1, 2
        Y = data.iloc[:, N_FEATURES].values - 1
        print(f"Arquivo '{data_path}' carregado com sucesso.")
    except Exception as e:
        print(f"Erro ao carregar o arquivo: {e}. Gerando dados sintéticos para demonstração.")
        from sklearn.datasets import make_classification
        X, Y = make_classification(n_samples=210, n_features=N_FEATURES, n_informative=5, n_redundant=0, n_classes=N_CLASSES, random_state=RANDOM_SEED)

    # Padronização dos dados (Importante para Regressão Logística)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Separação Treino/Teste para a avaliação final (Otimização será feita no conjunto de treino)
    X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=0.3, random_state=RANDOM_SEED)

    return X_train, X_test, Y_train, Y_test

X_train, X_test, Y_train, Y_test = load_and_preprocess_data()
print(f"Dados de Treino: X shape {X_train.shape}, Y shape {Y_train.shape}")
print(f"Dados de Teste: X shape {X_test.shape}, Y shape {Y_test.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Arquivo '/content/drive/MyDrive/Otimização/seeds_dataset.txt' carregado com sucesso.
Dados de Treino: X shape (147, 7), Y shape (147,)
Dados de Teste: X shape (63, 7), Y shape (63,)


#2. Implementação do Algoritmo Genético (DEAP):
O cromossomo agora é um vetor binário de tamanho $ N_{\text{FEATURES}}$ = 7.

O "1" na posição i significa que a feature i foi selecionada.





2.1 Funções GA

In [ ]:
# Definição da Estrutura de Fitness (Maximização da Acurácia)
# Nota: weights=(1.0,) significa que a função de avaliação deve retornar uma tupla de 1 valor.
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

# --- Função de Aptidão (Fitness Function) - CORRIGIDA ---
# Avalia a acurácia de um classificador usando as features selecionadas pelo indivíduo (vetor binário).
def evaluate_feature_selection(individual, X_train, Y_train, classifier=CLASSIFIER_FITNESS):
    features_index = np.where(np.array(individual) == 1)[0]

    # Se nenhuma feature for selecionada, retorna aptidão zero
    if len(features_index) == 0:
        # CORREÇÃO: Retorna uma tupla de 1 elemento (0.0,) para o GA.
        return 0.0,

    X_selected = X_train[:, features_index]

    # Avaliação robusta usando Cross-Validation (K-Fold = 3)
    kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    acc_scores = []

    for train_index, val_index in kf.split(X_selected):
        X_fold_train, X_fold_val = X_selected[train_index], X_selected[val_index]
        Y_fold_train, Y_fold_val = Y_train[train_index], Y_train[val_index]

        classifier.fit(X_fold_train, Y_fold_train)
        Y_pred = classifier.predict(X_fold_val)
        acc_scores.append(accuracy_score(Y_fold_val, Y_pred))

    # Fitness é a Acurácia média no K-Fold
    # CORREÇÃO: Retorna uma tupla de 1 elemento (Média,) para o GA e PSO.
    return np.mean(acc_scores),

# --- Inicialização GA (Toolbox DEAP) ---
toolbox_ga = base.Toolbox()
toolbox_ga.register("attr_bool", np.random.randint, 0, 2)
toolbox_ga.register("individual", tools.initRepeat, creator.Individual, toolbox_ga.attr_bool, n=N_FEATURES)
toolbox_ga.register("population", tools.initRepeat, list, toolbox_ga.individual)
# A função evaluate no DEAP agora retorna exatamente o que é esperado (1-tuple)
toolbox_ga.register("evaluate", evaluate_feature_selection, X_train=X_train, Y_train=Y_train, classifier=CLASSIFIER_FITNESS)

# --- Variação GA ---
toolbox_ga.register("select", tools.selTournament, tournsize=3)
toolbox_ga.register("mate", tools.cxUniform, indpb=0.5)
toolbox_ga.register("mutate", tools.mutFlipBit, indpb=MUT_RATE)

/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


#3. Otimização por Enxame de Partículas (PSO) para Seleção de Características

No PSO para seleção de características, a posição da partícula é um vetor de valores contínuos $[0, 1]$. O vetor é binarizado (ex: usando uma função sigmoide ou um simples arredondamento) para determinar se a feature será incluída ou não. Vamos usar o arredondamento para o PSO nativo: $\text{feature}_i = \text{round}(pos_i)$.


3.1 Funções PSO

In [ ]:
# Funções de suporte (init, update_velocity, update_position) não alteradas...

def init_swarm_pso(P, n_f):
    """Inicializa P partículas com posições (pos) e velocidades (vel) aleatórias."""
    pos = [np.random.uniform(0, 1, size=n_f) for _ in range(P)]
    vel = [np.random.uniform(-1, 1, size=n_f) for _ in range(P)]
    pos_best = [p.copy() for p in pos]
    return pos, vel, pos_best

def update_velocity(pos, vel, pos_best, global_best, w, c1, c2):
    """Calcula a nova velocidade da partícula."""
    vel_updated = []
    for i in range(len(pos)):
        r1 = np.random.uniform(0, 1)
        r2 = np.random.uniform(0, 1)
        vel_cognitive = c1 * r1 * (pos_best[i] - pos[i])
        vel_social = c2 * r2 * (global_best - pos[i])
        vel_updated.append(w * vel[i] + vel_cognitive + vel_social)
    return np.array(vel_updated)

def update_position(pos, vel, bounds):
    """Atualiza a posição da partícula e aplica limites [0, 1]."""
    pos_updated = []
    for i in range(len(pos)):
        pos_updated.append(np.clip(pos[i] + vel[i], a_min=bounds[0], a_max=bounds[1]))
    return pos_updated

def pso_feature_selection(swarm_size, c1, c2, w, X_train, Y_train, max_iters, classifier):
    """Loop principal do PSO para Seleção de Características."""
    n_features = X_train.shape[1]
    bounds = [0, 1]

    pos_swarm, vel_swarm, pos_best_swarm = init_swarm_pso(swarm_size, n_features)

    best_ind_cont = pos_swarm[0]
    best_ind_bin = np.round(best_ind_cont).astype(int)

    # CORREÇÃO: Desempacota o resultado de 1 elemento com a vírgula (,)
    global_best_fitness, = evaluate_feature_selection(list(best_ind_bin), X_train, Y_train, classifier)

    start_time_global = time.time()

    for iter in range(max_iters):
        for j, pos_cont in enumerate(pos_swarm):
            ind_bin = np.round(pos_cont).astype(int)
            # CORREÇÃO: Desempacota o resultado de 1 elemento com a vírgula (,)
            current_fitness, = evaluate_feature_selection(list(ind_bin), X_train, Y_train, classifier)

            pos_best_bin = np.round(pos_best_swarm[j]).astype(int)
            # CORREÇÃO: Desempacota o resultado de 1 elemento com a vírgula (,)
            score_known_best, = evaluate_feature_selection(list(pos_best_bin), X_train, Y_train, classifier)

            if current_fitness > score_known_best:
                pos_best_swarm[j] = pos_cont.copy()

            if current_fitness > global_best_fitness:
                global_best_fitness = current_fitness
                best_ind_cont = pos_cont.copy()

        for j in range(swarm_size):
            vel_swarm[j] = update_velocity(pos_swarm[j], vel_swarm[j], pos_best_swarm[j], best_ind_cont, w, c1, c2)
            pos_swarm[j] = update_position(pos_swarm[j], vel_swarm[j], bounds)

    end_time_global = time.time()

    final_best_ind_bin = np.round(best_ind_cont).astype(int)

    return list(final_best_ind_bin), (end_time_global - start_time_global), global_best_fitness

#4. Execução das Otimizações e Seleção de Características Finais
Vamos rodar os algoritmos de otimização uma única vez para obter o conjunto de características final otimizado.

In [ ]:
# --- 4.1. Execução do Algoritmo Genético (GA) - CORRIGIDA AQUI ---
print("\n--- 4.1. Execução do Algoritmo Genético (GA) para Seleção de Features ---")
# Cria a população inicial
pop_ga = toolbox_ga.population(n=SWARM_SIZE)
# Cria o Hall of Fame (armazena o melhor de todos)
hof_ga = tools.HallOfFame(1)

start_time_ga = time.time()
# O algoritmo eaSimple agora deve rodar corretamente, pois evaluate retorna 1 valor
pop_ga, log_ga = algorithms.eaSimple(pop_ga, toolbox_ga,
                                     cxpb=CX_RATE, mutpb=MUT_RATE,
                                     ngen=N_GENERATIONS, stats=None,
                                     halloffame=hof_ga, verbose=False)
end_time_ga = time.time()
time_ga_opt = end_time_ga - start_time_ga
best_features_ga = hof_ga[0]
# O fitness é um vetor (tupla) de 1 valor, acessado por [0]
best_acc_ga = hof_ga[0].fitness.values[0]

print(f"GA: Melhor Acurácia (no K-Fold Treino): {best_acc_ga:.4f}")
print(f"GA: Tempo de Otimização: {time_ga_opt:.4f}s")
print(f"GA: Features Selecionadas: {np.where(np.array(best_features_ga) == 1)[0].tolist()}")

# --- 4.2. Execução do PSO ---
print("\n--- 4.2. Execução do PSO para Seleção de Features ---")
# A função retorna: (features, tempo_otimização, acurácia_gbest)
best_features_pso, time_pso_opt, best_acc_pso = pso_feature_selection(
    swarm_size=SWARM_SIZE,
    c1=C1_PSO, c2=C2_PSO, w=W_PSO,
    X_train=X_train, Y_train=Y_train,
    max_iters=N_GENERATIONS,
    classifier=CLASSIFIER_FITNESS
)

print(f"PSO: Melhor Acurácia (no K-Fold Treino): {best_acc_pso:.4f}")
print(f"PSO: Tempo de Otimização: {time_pso_opt:.4f}s")
print(f"PSO: Features Selecionadas: {np.where(np.array(best_features_pso) == 1)[0].tolist()}")

# Define os 3 conjuntos de features que serão avaliados
feature_sets = {
    "ALL": np.array([1] * N_FEATURES),
    "GA_FS": np.array(best_features_ga),
    "PSO_FS": np.array(best_features_pso),
}


--- 4.1. Execução do Algoritmo Genético (GA) para Seleção de Features ---
GA: Melhor Acurácia (no K-Fold Treino): 0.9524
GA: Tempo de Otimização: 715.3998s
GA: Features Selecionadas: [0, 3, 4, 6]

--- 4.2. Execução do PSO para Seleção de Features ---
PSO: Melhor Acurácia (no K-Fold Treino): 0.9592
PSO: Tempo de Otimização: 2350.6509s
PSO: Features Selecionadas: [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 3, 3, 3, 3, 4, 4, 5, 5, 5, 5, 6, 6, 6, 6]


#5. Avaliação




In [ ]:
# --- 5. Treinamento e Avaliação dos Modelos (Repetição N_RUNS=30) ---
CLASSIFIERS = {
    "LogReg": LogisticRegression(random_state=RANDOM_SEED),
    "DecTree": DecisionTreeClassifier(random_state=RANDOM_SEED),
    "RandForest": RandomForestClassifier(random_state=RANDOM_SEED),
}

results_acc = {cls_name: {fs_name: [] for fs_name in feature_sets} for cls_name in CLASSIFIERS}
results_time = {cls_name: {fs_name: [] for fs_name in feature_sets} for cls_name in CLASSIFIERS}

print(f"\n--- 5. Treinamento e Avaliação (N={N_RUNS} Repetições) ---")

for run in range(N_RUNS):
    current_seed = RANDOM_SEED + run

    for cls_name, classifier in CLASSIFIERS.items():
        for fs_name, feature_vector in feature_sets.items():

            features_index = np.where(feature_vector == 1)[0]
            if len(features_index) == 0:
                 acc, tm = 0.0, 0.0
            else:
                X_train_selected = X_train[:, features_index]
                X_test_selected = X_test[:, features_index]

                classifier.set_params(random_state=current_seed)
                start_time = time.time()
                classifier.fit(X_train_selected, Y_train)
                end_time = time.time()

                Y_pred = classifier.predict(X_test_selected)
                acc = accuracy_score(Y_test, Y_pred)
                tm = end_time - start_time

            results_acc[cls_name][fs_name].append(acc)
            results_time[cls_name][fs_name].append(tm)

print("\n--- Resultados de Acurácia Média (no Conjunto de Teste) ---")
for cls_name in CLASSIFIERS:
    print(f"\nClassificador: **{cls_name}**")
    for fs_name in feature_sets:
        mean_acc = np.mean(results_acc[cls_name][fs_name])
        print(f"  - {fs_name}: {mean_acc:.4f} (Desvio Padrão: {np.std(results_acc[cls_name][fs_name]):.4f})")

# --- 6. Análise Estatística (Teste t) e Interpretação ---
def perform_t_test(results_A, results_B, model_A_name, model_B_name, metric_name, maximize=True):
    """Realiza o Teste t de Student (Welch) e interpreta o resultado."""
    results_A = np.array(results_A)
    results_B = np.array(results_B)

    t_stat, p_value = stats.ttest_ind(results_A, results_B, equal_var=False)

    print(f"\n--- Teste t: {model_A_name} vs {model_B_name} (Métrica: {metric_name}) ---")
    print(f"  Valor p: {p_value:.5f}")
    mean_A = np.mean(results_A)
    mean_B = np.mean(results_B)
    print(f"  Média {model_A_name}: {mean_A:.4f}, Média {model_B_name}: {mean_B:.4f}")

    if p_value < 0.05:
        if (maximize and mean_A > mean_B) or (not maximize and mean_A < mean_B):
            winner = model_A_name
        else:
            winner = model_B_name

        print(f"  Resultado: Há uma diferença estatisticamente significativa (**p < 0.05**). O melhor modelo é **{winner}**.")
    else:
        print("  Resultado: Não há diferença estatisticamente significativa (**p >= 0.05**).")
    return p_value

    print("\n--- 6.1. Análise de Comparação Estatística (Teste t) ---")

# Comparação 1: GA_FS vs PSO_FS para o Random Forest (Acurácia)
perform_t_test(
    results_acc["RandForest"]["GA_FS"],
    results_acc["RandForest"]["PSO_FS"],
    "RandForest (GA_FS)", "RandForest (PSO_FS)", "Acurácia", maximize=True
)

# Comparação 2: GA_FS vs ALL para a Regressão Logística (Acurácia)
perform_t_test(
    results_acc["LogReg"]["GA_FS"],
    results_acc["LogReg"]["ALL"],
    "LogReg (GA_FS)", "LogReg (ALL)", "Acurácia", maximize=True
)

# Comparação 3: ALL vs PSO_FS para a Árvore de Decisão (Tempo de Treinamento)
perform_t_test(
    results_time["DecTree"]["ALL"],
    results_time["DecTree"]["PSO_FS"],
    "DecTree (ALL)", "DecTree (PSO_FS)", "Tempo (Segundos)", maximize=False
)



--- 5. Treinamento e Avaliação (N=30 Repetições) ---

--- Resultados de Acurácia Média (no Conjunto de Teste) ---

Classificador: **LogReg**
  - ALL: 0.9048 (Desvio Padrão: 0.0000)
  - GA_FS: 0.8889 (Desvio Padrão: 0.0000)
  - PSO_FS: 0.9206 (Desvio Padrão: 0.0000)

Classificador: **DecTree**
  - ALL: 0.8450 (Desvio Padrão: 0.0255)
  - GA_FS: 0.8608 (Desvio Padrão: 0.0216)
  - PSO_FS: 0.8487 (Desvio Padrão: 0.0200)

Classificador: **RandForest**
  - ALL: 0.8894 (Desvio Padrão: 0.0126)
  - GA_FS: 0.8624 (Desvio Padrão: 0.0132)
  - PSO_FS: 0.9000 (Desvio Padrão: 0.0117)

--- Teste t: RandForest (GA_FS) vs RandForest (PSO_FS) (Métrica: Acurácia) ---
  Valor p: 0.00000
  Média RandForest (GA_FS): 0.8624, Média RandForest (PSO_FS): 0.9000
  Resultado: Há uma diferença estatisticamente significativa (**p < 0.05**). O melhor modelo é **RandForest (PSO_FS)**.

--- Teste t: LogReg (GA_FS) vs LogReg (ALL) (Métrica: Acurácia) ---
  Valor p: 0.00000
  Média LogReg (GA_FS): 0.8889, Média LogReg (A

/usr/local/lib/python3.12/dist-packages/scipy/stats/_axis_nan_policy.py:579: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


np.float64(4.0259660678272275e-08)